# Perception Flow & Evaluations

## Utilities

In [1]:
# Get the root path and data paths
#

from pathlib import Path


def repo_root(marker: str = "uv.lock") -> Path:
    """Nearest ancestor of the working directory containing *marker*."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / marker).is_file():
            return candidate
    raise FileNotFoundError(f"No {marker} found above {start}")

DATA_IN = repo_root() / "data_in"



## Runs

In [2]:
# Get the test text and ground-truth affect
#

import pandas as pd

data_file = Path(DATA_IN / "benchmark_text.csv")
df_benchmark = pd.read_csv(data_file,
                           dtype={"text": "string", "basic4/1": "category", "ekman6/1": "category"})

df_benchmark.info() 
display(df_benchmark.describe())
display(df_benchmark.sample(5))


<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   text      20 non-null     string  
 1   basic4/1  17 non-null     category
 2   ekman6/1  17 non-null     category
dtypes: category(2), string(1)
memory usage: 756.0 bytes


,text,basic4/1,ekman6/1
count,20,17,17
unique,20,4,6
top,I am so happy,fear_surprise,happiness
freq,1,5,4


,text,basic4/1,ekman6/1
9,I'm scared of what happens next,fear_surprise,fear
3,I feel miserable today,sadness,sadness
12,I just got the job!,happiness,happiness
11,that was completely unexpected,fear_surprise,surprise
17,what time is the meeting,NaN,NaN


In [3]:
# Decode the sentences
#

from dataclasses import asdict

from asa.core.affect import AffectVector, Utterance
from asa.core.representations import BASIC4, EKMAN6
from asa.perception.decode_keyword import BASIC4_KEYWORDS, EKMAN6_KEYWORDS, KeywordDecoder

basic4_decoder = KeywordDecoder(representation=BASIC4, table=BASIC4_KEYWORDS)
label_col = BASIC4.id
blank_values = {axis: BASIC4.rest for axis in BASIC4.axes}

# observations = []
results = []
for row in df_benchmark.to_dict("records"):
    print(row)
    label = row[label_col]
    # intended_affect = None
    # if not pd.isna(label):
    #     intended_affect = AffectVector(representation=BASIC4.id, values= blank_values | {label: 1.0})
    values = blank_values if pd.isna(label) else blank_values | {label: 1.0}
    intended_affect = AffectVector(representation=BASIC4.id, values=values)
    utterance = Utterance(text=row["text"], source="input:benchmark", intended=intended_affect)
    print(utterance)
    affect_observation = await basic4_decoder.decode(utterance)
    print(affect_observation)
    # observations.append(affect_observation)

    results.append({"utterance": asdict(utterance), "observation": asdict(affect_observation)})

flat_results = pd.json_normalize(results)
# flat_df = pd.json_normalize([asdict(o) for o in observations])

{'text': 'I am so happy', 'basic4/1': 'happiness', 'ekman6/1': 'happiness'}
Utterance(text='I am so happy', source='input:benchmark', intended=AffectVector(representation='basic4/1', values={<FourEmotions.HAPPINESS: 'happiness'>: 1.0, <FourEmotions.SADNESS: 'sadness'>: 0.0, <FourEmotions.FEAR_SURPRISE: 'fear_surprise'>: 0.0, <FourEmotions.ANGER_DISGUST: 'anger_disgust'>: 0.0}), id='2bdec0fc3cbf', at=datetime.datetime(2026, 8, 5, 0, 5, 49, 188918, tzinfo=datetime.timezone.utc), schema='utterance/1')
AffectEvidence(target=<Target.OTHER: 'other'>, affect=AffectVector(representation='basic4/1', values={<FourEmotions.HAPPINESS: 'happiness'>: 0.7, <FourEmotions.SADNESS: 'sadness'>: 0.0, <FourEmotions.FEAR_SURPRISE: 'fear_surprise'>: 0.0, <FourEmotions.ANGER_DISGUST: 'anger_disgust'>: 0.0}), confidence=None, source='decoder:rule', rationale='matched: happiness=happy', computed_from=None, of_input='2bdec0fc3cbf', at=datetime.datetime(2026, 8, 5, 0, 5, 49, 188918, tzinfo=datetime.timezone.utc),